# Fundamental SPP — GPU sweep on Colab

End-to-end reproduction of *Fundamental Information for Low-Turnover Equity ML*.
This notebook resumes the 60-config walk-forward sweep (4 architectures x 3 regimes x 5 years)
on a Colab GPU, picking up where the CPU box left off (8/60 configs already done).

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU** (T4 is fine).

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Set Runtime -> Change runtime type -> GPU.'
!nvidia-smi -L

## 1. Get the code

In [ ]:
import os
if not os.path.exists('Fundamental_SPP'):
    !git clone https://github.com/ShadowKingYT444/Fundamental_SPP
else:
    !git -C Fundamental_SPP pull --ff-only
%cd Fundamental_SPP
!git log --oneline -1

## 2. Dependencies
Colab already ships torch (CUDA), pandas, numpy, scipy, scikit-learn and matplotlib. Only pyarrow (parquet) may be missing.

In [ ]:
!pip install -q pyarrow

## 3. Stage the data from Google Drive
The ~550 MB prepared dataset (panels, prices) plus the 8 finished checkpoints/scores/results were packed into one tarball on Drive. This mounts Drive and extracts it into the repo layout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, tarfile
SRC = '/content/drive/MyDrive/fundamental_spp_data'
TARBALL = os.path.join(SRC, 'sweep_data.tar.gz')
assert os.path.isfile(TARBALL), f'Data tarball not found: {TARBALL}'
with tarfile.open(TARBALL, 'r:gz') as t:
    members = [m for m in t.getmembers() if m.isfile()]
    t.extractall('.')
print(f'extracted {len(members)} files')
!ls -lh data/ | head -12

In [ ]:
import os, shutil
SRC = '/content/drive/MyDrive/fundamental_spp_data/colab_out'
restored = 0
for d in ['checkpoints', 'scores', 'results']:
    src_d = os.path.join(SRC, d)
    if not os.path.isdir(src_d):
        print(f'no {src_d} on Drive, skipping'); continue
    os.makedirs(d, exist_ok=True)
    for f in os.listdir(src_d):
        if f == '.run_next.lock':
            continue  # stale lock from a dead runtime; never restore it
        s, dd = os.path.join(src_d, f), os.path.join(d, f)
        if os.path.isfile(s) and (not os.path.exists(dd)
                or os.path.getmtime(s) > os.path.getmtime(dd)):
            shutil.copy2(s, dd); restored += 1
lock = os.path.join('results', '.run_next.lock')
if os.path.exists(lock):
    os.remove(lock); print('removed stale', lock)
print(f'restored {restored} files from Drive colab_out')
import json as _j
_d = _j.load(open('results/run_status.json'))
_ok = [k for k, v in _d.items() if isinstance(v, dict) and v.get('status') == 'ok']
print(len(_ok), 'configs already ok')


## 4. Smoke test: 1 epoch of MISS on the GPU
Uses a throwaway checkpoint dir so the real sweep state is untouched. Watch for `device=cuda` in the log and a sensible per-batch time.

In [ ]:
!python -u src/run_train.py --model miss --regime tech63 --year 2021 --epochs 1 --device cuda --checkpoint-dir /tmp/smoke_ckpt --seed 123 2>&1 | tail -6

## 5. Run the sweep (resumable)\nResumes from run_status.json (13/60 as of Sept 23; driver skips finished configs): the two blocked long configs (`miss_tech63_2024/2025`) run at full batch 512 on the GPU, then everything else. Fully resumable — if Colab disconnects, just re-run this cell. A background thread copies every finished checkpoint/score/result to Drive every 15 minutes, so a runtime loss can't wipe more than one config. Budget is ~11h; Colab caps sessions around 12h.

In [ ]:
import subprocess, threading, time, os, shutil, sys

OUT = '/content/drive/MyDrive/fundamental_spp_data/colab_out'
os.makedirs(OUT, exist_ok=True)

def _sync_once():
    for d in ['checkpoints', 'scores', 'results']:
        if not os.path.isdir(d): continue
        dst = os.path.join(OUT, d); os.makedirs(dst, exist_ok=True)
        for f in os.listdir(d):
            s, dd = os.path.join(d, f), os.path.join(dst, f)
            if os.path.isfile(s) and (not os.path.exists(dd)
                    or os.path.getmtime(s) > os.path.getmtime(dd)):
                shutil.copy2(s, dd)

stop = False
def sync_loop():
    while not stop:
        time.sleep(900)  # every 15 min
        _sync_once()
        print('checkpoint synced to Drive at', time.strftime('%H:%M:%S'), flush=True)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
try:
    r = subprocess.run([sys.executable, '-u', 'src/run_next.py',
                        '--device', 'cuda', '--allow-long',
                        '--time-budget', '40000'])
    print('run_next exit:', r.returncode)
finally:
    stop = True
_sync_once()
print('sweep cell done')


## 6. Evaluate + figures (after 60/60)
Once the sweep prints DONE, build the metrics, tables and paper figures.

In [ ]:
!python -u src/evaluate.py 2>&1 | tail -20
!python -u src/make_figures.py 2>&1 | tail -5

## 7. Sync everything back to Drive
Checkpoints, scores, metrics and figures land in `fundamental_spp_data/colab_out/` for download.

In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/fundamental_spp_data/colab_out'
os.makedirs(OUT, exist_ok=True)
for d in ['checkpoints', 'scores', 'results']:
    dst = os.path.join(OUT, d); os.makedirs(dst, exist_ok=True)
    if os.path.isdir(d):
        for f in os.listdir(d):
            s = os.path.join(d, f)
            if os.path.isfile(s):
                shutil.copy2(s, dst)
print('final sync done')
!du -sh /content/drive/MyDrive/fundamental_spp_data/colab_out
